In [1]:
import sys
import os

# Get the absolute path to the project root (the folder that contains 'lattice')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

In [2]:
from matchms.importing import load_from_mgf
from lattice.preprocessing.SpectrumFilter import SpectrumFilter

In [ ]:
yaml_path = ("../config.yaml")  # or None if you want defaults, here we provide a custom config file

In [4]:
spectra_list = list(load_from_mgf("/Users/rtlortega/Documents/PhD/Datasets/GNPS-NIH-NATURALPRODUCTSLIBRARY.mgf"))
print(f"Loaded {len(spectra_list)} spectra from MGF.")


Loaded 1267 spectra from MGF.


In [5]:
processor = SpectrumFilter(spectra=spectra_list, yaml_config_path=yaml_path)
filtered_spectra = processor.process()


→ After default filters: 1267 spectra remain
→ After select_by_mz: 1267 spectra remain
→ After select_by_relative_intensity: 1267 spectra remain
→ After require_minimum_number_of_peaks: 1217 spectra remain
→ After normalize_intensities: 1217 spectra remain
Final spectra count: 1217


In [ ]:
print(f"Number of spectra after filtering: {len(filtered_spectra)}")

Number of spectra after filtering: 1217


In [7]:
from lattice.scores.SimilarityCalculator import SimilarityCalculator

In [8]:
# Initialize
calculator = SimilarityCalculator(filtered_spectra)

In [ ]:

# Spec2Vec
#calculator.calculate_modcosine(tolerance=0.0.01)

In [ ]:

# Spec2Vec
calculator.load_spec2vec('/Users/rtlortega/Documents/PhD/Spec2Vec/model_positive_mode/positive_train_data/150225_Spec2Vec_pos_CleanedLibraries.model')
spec2vec_scores = calculator.calculate_spec2vec()  


In [ ]:
# MS2DeepScore if you want
#calculator.load_ms2deepscore('../ms2deepscore_model.pt')
#ms2ds_scores = calculator.calculate_ms2deepscore()  # 

In [ ]:
from lattice.networking.SimilarityNetworkMod import SimilarityNetworkMod

ms_network = SimilarityNetworkMod(
        identifier_key="scans",
        score_cutoff=0.7,
        max_links=10,
        min_peaks=None, #change if you use mod cosine
        link_method='single'  # try 'mutual' too to see difference
    )

In [ ]:
ms_network.create_network(spec2vec_scores, score_name="Spec2Vec") #scorename can change
ms_network.filter_components(max_component_size=100)
ms_network.export_to_graphml(filename="NP_spec2vec_network.graphml")